# Reproducible ARGO(X) → humidity-forced SEIRS–EAKF → weekly $R_t$ pipeline

This notebook is a clean, auditable version of the analysis path used to derive the weekly state-level effective reproduction number estimates used downstream in the influenza mobility/transmission analysis.

**Scope of this notebook**

1. Load finalized ARGO(X) step-2 weekly predictions.
2. Load daily and/or weekly absolute humidity (AH) files.
3. Align ARGO(X) observations and AH to a Monday weekly grid.
4. Run a daily-step SEIRS compartmental simulation with weekly EAKF assimilation.
5. Export weekly $R_t$ estimates in long and wide format.

**Not included here:** mobility analyses, figure-specific smoothing, Spearman correlation calculations, or manuscript figures.


## 0. Model summary

The final $R_t$ estimates are generated from a **humidity-forced SEIRS compartmental model with weekly Ensemble Adjustment Kalman Filter (EAKF) assimilation**.

The model state is:

\[
X = [S, E, I, R, \log(\beta_{ref}), c_{scale}]
\]

Daily dynamics follow:

\[
\begin{aligned}
\text{new exposed} &= \beta_t S I \\
\text{new infected} &= \sigma E \\
\text{new recovered} &= \gamma I \\
\text{waned} &= \omega R
\end{aligned}
\]

Weekly ARGO(X) step-2 predictions serve as the observation stream used to constrain latent epidemic states through the EAKF update.


In [1]:
# ==========================================================
# 1. Imports and project paths
# ==========================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

# ---- USER EDIT: project root on local machine ----
# For Ben's desktop, leave this as-is. For a clean GitHub repo, set PROJECT_ROOT
# to the repository root or leave it as Path.cwd().
PROJECT_ROOT = Path("/Users/benjamincristol/Desktop/dissertation/argo_seir_eakf")

# Common directories searched automatically. This makes the notebook runnable both
# from Ben's working directory and from a clean repo with ./data and ./outputs folders.
CACHE = PROJECT_ROOT / "cache"
OUT = PROJECT_ROOT / "out" / "argox"
DATA = Path.cwd() / "data"
OUTPUTS = Path.cwd() / "outputs"
LOCAL = Path.cwd()

ARGO_STEP2_CANDIDATES = [
    OUT / "state_preds_argo_step2_long.csv",
    OUT / "state_preds_argo_step2.csv",
    CACHE / "state_preds_argo_step2_long.csv",
    CACHE / "state_preds_argo_step2.csv",
    DATA / "state_preds_argo_step2_long.csv",
    DATA / "state_preds_argo_step2.csv",
    LOCAL / "state_preds_argo_step2_long.csv",
    LOCAL / "state_preds_argo_step2.csv",
]

AH_DAILY_CANDIDATES = [
    CACHE / "ah_daily_allstates.csv",
    DATA / "ah_daily_allstates.csv",
    LOCAL / "ah_daily_allstates.csv",
]

AH_WEEKLY_CANDIDATES = [
    CACHE / "ah_weekly_WMON_aligned.csv",
    DATA / "ah_weekly_WMON_aligned.csv",
    LOCAL / "ah_weekly_WMON_aligned.csv",
]

# Outputs default to Ben's cache if that project folder exists; otherwise use ./outputs.
OUTPUT_DIR = CACHE if CACHE.exists() else OUTPUTS
RT_LONG_OUT = OUTPUT_DIR / "rt_state_weekly.csv"
RT_WIDE_OUT = OUTPUT_DIR / "Rt_weekly_wide.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CACHE:", CACHE)
print("OUT:", OUT)
print("DATA:", DATA)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf
CACHE: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/cache
OUT: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/out/argox
DATA: /Users/benjamincristol/Desktop/dissertation/data
OUTPUT_DIR: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/cache


In [2]:
# ==========================================================
# 2. Utility functions for loading and alignment
# ==========================================================

STATE_ABBRS = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","IA","ID","IL","IN","KS","KY","LA",
    "MA","MD","ME","MI","MN","MO","MS","MT","NC","ND","NE","NH","NJ","NM","NV","NY","OH","OK",
    "OR","PA","RI","SC","SD","TN","TX","UT","VA","VT","WA","WI","WV","WY","DC"
}

def first_existing(paths, label="file"):
    """Return the first existing path from a list, with a clear error if none exist."""
    checked = []
    for p in paths:
        p = Path(p)
        checked.append(str(p))
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Could not find required {label}. Checked these candidate paths:\n" + "\n".join(checked)
    )

def monday_week_start(x):
    """Convert dates to Monday week-start timestamps."""
    dt = pd.to_datetime(x)
    if isinstance(dt, pd.Series):
        return dt - pd.to_timedelta(dt.dt.weekday, unit="D")
    if isinstance(dt, pd.DatetimeIndex):
        return dt - pd.to_timedelta(dt.weekday, unit="D")
    return dt - pd.to_timedelta(dt.weekday(), unit="D")

def standardize_state_cols(df):
    """Uppercase state-abbreviation columns while leaving non-state columns alone."""
    out = df.copy()
    out.columns = [str(c).strip().upper() if str(c).strip().upper() in STATE_ABBRS else str(c).strip() for c in out.columns]
    return out

def read_argox_step2(path):
    """Read ARGO(X) step-2 predictions in long or wide CSV format and return wide date x state."""
    raw = pd.read_csv(path)
    cols_lower = {c.lower(): c for c in raw.columns}

    # Long format: date, state, pred/prediction/y_pred or first numeric value column.
    if {"date", "state"}.issubset(cols_lower):
        date_col = cols_lower["date"]
        state_col = cols_lower["state"]
        pred_col = cols_lower.get("pred") or cols_lower.get("prediction") or cols_lower.get("y_pred")
        if pred_col is None:
            numeric_cols = [
                c for c in raw.columns
                if c not in {date_col, state_col} and pd.api.types.is_numeric_dtype(raw[c])
            ]
            if not numeric_cols:
                raise ValueError("Could not identify prediction column in ARGO long file.")
            pred_col = numeric_cols[0]
        df = raw[[date_col, state_col, pred_col]].copy()
        df.columns = ["date", "state", "pred"]
        df["date"] = monday_week_start(df["date"])
        df["state"] = df["state"].astype(str).str.upper().str.strip()
        wide = df.pivot_table(index="date", columns="state", values="pred", aggfunc="mean").sort_index()
        wide.columns.name = None
        return standardize_state_cols(wide.apply(pd.to_numeric, errors="coerce"))

    # Wide format with explicit date column.
    if "date" in cols_lower:
        date_col = cols_lower["date"]
        df = raw.copy()
        df[date_col] = monday_week_start(df[date_col])
        df = df.set_index(date_col).sort_index()
        return standardize_state_cols(df.apply(pd.to_numeric, errors="coerce"))

    # Wide format with date as first/index column.
    df = pd.read_csv(path, index_col=0)
    df.index = monday_week_start(pd.DatetimeIndex(df.index))
    df = df.sort_index()
    return standardize_state_cols(df.apply(pd.to_numeric, errors="coerce"))

def read_ah_wide(path):
    """Read daily or weekly AH wide CSV with date column or date index."""
    df = pd.read_csv(path)
    cols_lower = {c.lower(): c for c in df.columns}
    if "date" in cols_lower:
        date_col = cols_lower["date"]
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)
    else:
        df = pd.read_csv(path, index_col=0)
        df.index = pd.to_datetime(df.index)
    return standardize_state_cols(df).sort_index().apply(pd.to_numeric, errors="coerce")

def pack_daily_ah_by_week(ah_daily, weeks, states):
    """Pack each Monday week into a length-7 daily AH vector for each state."""
    out = pd.DataFrame(index=pd.DatetimeIndex(weeks), columns=states, dtype=object)
    ah_daily = ah_daily.sort_index()
    for wk in out.index:
        days = pd.date_range(wk, periods=7, freq="D")
        block = ah_daily.reindex(days).interpolate(limit=2).ffill().bfill()
        for st in states:
            if st in block.columns:
                arr = np.asarray(block[st], dtype=float)
                if np.isfinite(arr).sum() == 0:
                    arr = np.full(7, np.nan)
            else:
                arr = np.full(7, np.nan)
            out.at[wk, st] = arr
    return out


In [3]:
# ==========================================================
# 3. Load finalized ARGO(X) step-2 predictions and AH files
# ==========================================================

ARGO_STEP2_CSV = first_existing(ARGO_STEP2_CANDIDATES, label="ARGO(X) step-2 prediction CSV")
AH_DAILY_CSV = first_existing(AH_DAILY_CANDIDATES, label="daily AH CSV")
AH_WEEKLY_CSV = first_existing(AH_WEEKLY_CANDIDATES, label="weekly AH CSV")

print("Using ARGO(X) step-2 predictions:", ARGO_STEP2_CSV)
print("Using daily AH:", AH_DAILY_CSV)
print("Using weekly AH:", AH_WEEKLY_CSV)

preds_wide = read_argox_step2(ARGO_STEP2_CSV).sort_index()
ah_daily = read_ah_wide(AH_DAILY_CSV)
ah_weekly = read_ah_wide(AH_WEEKLY_CSV)
ah_weekly.index = monday_week_start(pd.DatetimeIndex(ah_weekly.index))
ah_weekly = ah_weekly.groupby(level=0).mean().sort_index()

states = sorted(set(preds_wide.columns) & set(ah_daily.columns))
if not states:
    raise ValueError("No overlapping state columns were found between ARGO predictions and daily AH.")

preds_wide = preds_wide[states]
ah_daily = ah_daily[states]
ah_weekly = ah_weekly[[s for s in states if s in ah_weekly.columns]]

print("ARGO preds:", preds_wide.shape, preds_wide.index.min().date(), "→", preds_wide.index.max().date())
print("Daily AH:", ah_daily.shape, ah_daily.index.min().date(), "→", ah_daily.index.max().date())
print("Weekly AH:", ah_weekly.shape, ah_weekly.index.min().date(), "→", ah_weekly.index.max().date())
print("States:", len(states), states[:10], "...")


Using ARGO(X) step-2 predictions: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/out/argox/state_preds_argo_step2_long.csv
Using daily AH: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/cache/ah_daily_allstates.csv
Using weekly AH: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/cache/ah_weekly_WMON_aligned.csv
ARGO preds: (467, 49) 2016-10-10 → 2025-09-15
Daily AH: (3270, 49) 2016-10-03 → 2025-09-15
Weekly AH: (468, 49) 2016-10-03 → 2025-09-15
States: 49 ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DE', 'GA', 'HI'] ...


In [4]:
# ==========================================================
# 4. Align ARGO observations and daily AH windows
# ==========================================================

weeks = pd.DatetimeIndex(preds_wide.index).sort_values()
ah_daily_per_week = pack_daily_ah_by_week(ah_daily, weeks, states)

if ah_weekly is not None:
    ah_weekly_aligned = ah_weekly.reindex(weeks).interpolate(limit=1).ffill().bfill()
else:
    ah_weekly_aligned = None

for st in states:
    state_median = float(np.nanmedian(ah_daily[st].values)) if st in ah_daily.columns else 0.008
    for wk in weeks:
        arr = np.asarray(ah_daily_per_week.at[wk, st], dtype=float)
        if not np.isfinite(arr).all():
            if ah_weekly_aligned is not None and st in ah_weekly_aligned.columns and np.isfinite(ah_weekly_aligned.at[wk, st]):
                fill = float(ah_weekly_aligned.at[wk, st])
            else:
                fill = state_median
            ah_daily_per_week.at[wk, st] = np.where(np.isfinite(arr), arr, fill)

print("Weekly observation grid:", weeks.min().date(), "→", weeks.max().date(), "| weeks =", len(weeks))
print("Packed daily AH windows:", ah_daily_per_week.shape)


Weekly observation grid: 2016-10-10 → 2025-09-15 | weeks = 467
Packed daily AH windows: (467, 49)


## 5. SEIRS + EAKF implementation

This block is self-contained so a colleague can inspect or replace the model mechanics without tracing older development notebooks.

Default model settings mirror the v10 fastpath implementation:

| Setting | Default |
|---|---:|
| Ensemble members | 200 |
| Inflation | 1.05 |
| Observation variance | $0.05^2$ |
| Process variance | $0.10^2$ |
| Random seed | 42 |

Default priors/starting values include `S=0.99`, `E=0.005`, `I=0.005`, `L=1/2`, `D=1/3`, `W=1/180`, `R0=1.5`, `k_hum=0.08`, and `AH_ref=0.008`.


In [5]:
# ==========================================================
# 5. SEIRS + EAKF functions
# ==========================================================

def sample_from_priors(rng, priors=None):
    priors = priors or {}
    return {
        "S": priors.get("S", 0.99),
        "E": priors.get("E", 0.005),
        "I": priors.get("I", 0.005),
        "L": priors.get("L", 1/2.0),
        "D": priors.get("D", 1/3.0),
        "W": priors.get("W", 1/180.0),
        "R0": priors.get("R0", 1.5),
        "k_hum": priors.get("k_hum", 0.08),
        "AH_ref": priors.get("AH_ref", 0.008),
    }

def seirs_daily_step(S, E, I, R, beta_t, sigma, gamma, omega):
    new_exposed = beta_t * S * I
    new_infected = sigma * E
    new_recovered = gamma * I
    waned = omega * R
    S2 = S - new_exposed + waned
    E2 = E + new_exposed - new_infected
    I2 = I + new_infected - new_recovered
    R2 = R + new_recovered - waned
    return S2, E2, I2, R2

def eakf_adjust(X, h, y, R_var, infl, rng):
    N = X.shape[0]
    Xm = X - X.mean(axis=0, keepdims=True)
    Xp = X.mean(axis=0, keepdims=True) + np.sqrt(infl) * Xm
    h = np.asarray(h, dtype=float)
    h_m = float(np.nanmean(h))
    h_anom = h - h_m
    Pxh = (Xp - Xp.mean(axis=0, keepdims=True)).T @ h_anom / max(N - 1, 1)
    Phh = float(np.nanvar(h, ddof=1)) if N > 1 else float(np.nanvar(h))
    K = Pxh / (Phh + R_var + 1e-8)
    Xa = Xp + float(y - h_m) * K
    Xa += rng.normal(0.0, 1e-4, size=Xa.shape)
    return Xa

def run_eakf_weekly(y_obs_weekly, ah_daily_per_week, priors=None, config=None):
    config = config or dict(N_ens=200, inflation=1.05, obs_var=0.05**2, proc_var=0.10**2, seed=42)
    rng = np.random.default_rng(int(config.get("seed", 42)))
    N = int(config.get("N_ens", 200))
    infl = float(config.get("inflation", 1.05))
    obs_v = float(config.get("obs_var", 0.05**2))
    proc_v = float(config.get("proc_var", 0.10**2))

    y = y_obs_weekly.astype(float).replace([np.inf, -np.inf], np.nan).ffill().bfill()
    if y.abs().max() > 100:
        yg = y.pct_change().replace([np.inf, -np.inf], np.nan).fillna(0.0)
        y = np.log1p(yg)
    y_mu = float(y.mean())
    y_sd = float(y.std(ddof=1)) if float(y.std(ddof=1)) > 0 else 1.0
    y_std = (y - y_mu) / y_sd
    weeks = pd.DatetimeIndex(y.index)

    pars = sample_from_priors(rng, priors)
    gamma = 1.0 / max(float(pars["D"]), 0.5)
    sigma = 1.0 / max(float(pars["L"]), 0.5)
    omega = 1.0 / max(float(pars["W"]), 1.0)
    k_hum = float(pars["k_hum"])
    AH_ref = float(pars["AH_ref"])

    S = rng.normal(float(pars["S"]), 0.02, size=N)
    S = np.clip(S, 0.01, 0.999)
    E = rng.lognormal(mean=np.log(1e-6), sigma=0.5, size=N)
    I = rng.lognormal(mean=np.log(1e-6), sigma=0.5, size=N)
    R = np.clip(1.0 - S - E - I, 0.0, 1.0)
    log_beta_ref = np.log(float(pars["R0"]) * gamma) + rng.normal(0.0, 0.05, size=N)
    c_scale = rng.normal(1.0, 0.5, size=N)
    X = np.vstack([S, E, I, R, log_beta_ref, c_scale]).T

    Rt_out = []
    for wk in weeks:
        ah7 = np.asarray(ah_daily_per_week.loc[wk, "ah7"], dtype=float) if wk in ah_daily_per_week.index else np.full(7, AH_ref)
        if ah7.size != 7:
            pad = np.full(7, np.nanmean(ah7) if ah7.size else AH_ref, dtype=float)
            pad[:min(ah7.size, 7)] = ah7[:min(ah7.size, 7)]
            ah7 = pad
        ah7 = np.where(np.isfinite(ah7), ah7, AH_ref)

        beta_week = np.zeros(N)
        S_week = np.zeros(N)
        for d in range(7):
            beta_t = np.exp(X[:, 4]) * np.exp(k_hum * (AH_ref - float(ah7[d])))
            S, E, I, R = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
            S2, E2, I2, R2 = seirs_daily_step(S, E, I, R, beta_t, sigma, gamma, omega)
            noise = rng.normal(0.0, np.sqrt(proc_v), size=(N, 4))
            S2 = np.clip(S2 + noise[:, 0], 0.0, 1.0)
            E2 = np.clip(E2 + noise[:, 1], 0.0, 1.0)
            I2 = np.clip(I2 + noise[:, 2], 0.0, 1.0)
            R2 = np.clip(R2 + noise[:, 3], 0.0, 1.0)
            tot = S2 + E2 + I2 + R2
            tot[tot == 0] = 1.0
            S2, E2, I2, R2 = S2/tot, E2/tot, I2/tot, R2/tot
            X[:, 0], X[:, 1], X[:, 2], X[:, 3] = S2, E2, I2, R2
            beta_week += beta_t
            S_week += S2

        beta_week /= 7.0
        S_week /= 7.0
        h = X[:, 5] * (sigma * X[:, 1])

        if wk in y_std.index and np.isfinite(y_std.loc[wk]):
            X = eakf_adjust(X, h=h, y=float(y_std.loc[wk]), R_var=obs_v, infl=infl, rng=rng)

        X[:, 0:4] = np.clip(X[:, 0:4], 0.0, 1.0)
        tot = X[:, 0] + X[:, 1] + X[:, 2] + X[:, 3]
        X[:, 0:4] = (X[:, 0:4].T / np.where(tot > 0, tot, 1.0)).T
        X[:, 4] += rng.normal(0.0, 0.01, size=N)
        X[:, 5] += rng.normal(0.0, 0.05, size=N)
        Rt_week = (float(np.mean(beta_week)) / gamma) * float(np.mean(S_week))
        Rt_out.append(np.clip(Rt_week, 0.3, 3.5))

    return pd.Series(Rt_out, index=weeks, name="rt")


In [6]:
# ==========================================================
# 6. Run SEIRS + EAKF for each state
# ==========================================================

CONFIG = dict(N_ens=200, inflation=1.05, obs_var=0.05**2, proc_var=0.10**2, seed=42)
PRIORS = dict(S=0.99, E=0.005, I=0.005, L=1/2.0, D=1/3.0, W=1/180.0, R0=1.5, k_hum=0.08, AH_ref=0.008)

rt_by_state = {}
for i, st in enumerate(states, start=1):
    y = preds_wide[st].dropna()
    if y.empty:
        print(f"[{i:02d}/{len(states)}] {st}: skipped, no ARGO observations")
        continue
    ah_state = pd.DataFrame({"ah7": ah_daily_per_week.loc[y.index, st]})
    cfg = dict(CONFIG)
    cfg["seed"] = CONFIG["seed"] + i
    rt_by_state[st] = run_eakf_weekly(y_obs_weekly=y, ah_daily_per_week=ah_state, priors=PRIORS, config=cfg)
    print(f"[{i:02d}/{len(states)}] {st}: {len(rt_by_state[st])} weeks")

rt_wide = pd.DataFrame(rt_by_state).sort_index()
rt_wide.index.name = "date"
rt_long = (rt_wide.reset_index().melt(id_vars="date", var_name="state", value_name="rt").dropna().sort_values(["state", "date"]).reset_index(drop=True))

print("rt_wide:", rt_wide.shape, rt_wide.index.min().date(), "→", rt_wide.index.max().date())
print("rt_long:", rt_long.shape)
rt_long.head()


/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:89: RuntimeWarning: overflow encountered in exp
  beta_t = np.exp(X[:, 4]) * np.exp(k_hum * (AH_ref - float(ah7[d])))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:20: RuntimeWarning: invalid value encountered in multiply
  new_exposed = beta_t * S * I
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:35: RuntimeWarning: Mean of empty slice
  h_m = float(np.nanmean(h))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:38: RuntimeWarning: Degrees of freedom <= 0 for slice.
  Phh = float(np.nanvar(h, ddof=1)) if N > 1 else float(np.nanvar(h))


[01/49] AK: 467 weeks


/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:89: RuntimeWarning: overflow encountered in exp
  beta_t = np.exp(X[:, 4]) * np.exp(k_hum * (AH_ref - float(ah7[d])))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:20: RuntimeWarning: invalid value encountered in multiply
  new_exposed = beta_t * S * I
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:35: RuntimeWarning: Mean of empty slice
  h_m = float(np.nanmean(h))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:38: RuntimeWarning: Degrees of freedom <= 0 for slice.
  Phh = float(np.nanvar(h, ddof=1)) if N > 1 else float(np.nanvar(h))


[02/49] AL: 467 weeks


/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:89: RuntimeWarning: overflow encountered in exp
  beta_t = np.exp(X[:, 4]) * np.exp(k_hum * (AH_ref - float(ah7[d])))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:20: RuntimeWarning: invalid value encountered in multiply
  new_exposed = beta_t * S * I
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:35: RuntimeWarning: Mean of empty slice
  h_m = float(np.nanmean(h))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:38: RuntimeWarning: Degrees of freedom <= 0 for slice.
  Phh = float(np.nanvar(h, ddof=1)) if N > 1 else float(np.nanvar(h))


...

[47/49] WI: 467 weeks


/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:89: RuntimeWarning: overflow encountered in exp
  beta_t = np.exp(X[:, 4]) * np.exp(k_hum * (AH_ref - float(ah7[d])))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:20: RuntimeWarning: invalid value encountered in multiply
  new_exposed = beta_t * S * I
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:35: RuntimeWarning: Mean of empty slice
  h_m = float(np.nanmean(h))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:38: RuntimeWarning: Degrees of freedom <= 0 for slice.
  Phh = float(np.nanvar(h, ddof=1)) if N > 1 else float(np.nanvar(h))


[48/49] WV: 467 weeks
[49/49] WY: 467 weeks
rt_wide: (467, 49) 2016-10-10 → 2025-09-15
rt_long: (16636, 3)


/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:89: RuntimeWarning: overflow encountered in exp
  beta_t = np.exp(X[:, 4]) * np.exp(k_hum * (AH_ref - float(ah7[d])))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:20: RuntimeWarning: invalid value encountered in multiply
  new_exposed = beta_t * S * I
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:35: RuntimeWarning: Mean of empty slice
  h_m = float(np.nanmean(h))
/var/folders/yr/9nvl5l8149n_vtxp4r8pmw6r0000gn/T/ipykernel_48310/1935520036.py:38: RuntimeWarning: Degrees of freedom <= 0 for slice.
  Phh = float(np.nanvar(h, ddof=1)) if N > 1 else float(np.nanvar(h))


,date,state,rt
0,2016-10-10,AK,0.619384
1,2016-10-17,AK,0.300000
2,2016-10-24,AK,0.339740
3,2016-10-31,AK,0.300000
4,2016-11-07,AK,0.319596


In [7]:
# ==========================================================
# 7. Export reproducible Rt outputs
# ==========================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rt_long.to_csv(RT_LONG_OUT, index=False)
rt_wide.to_csv(RT_WIDE_OUT)
print("Wrote:", RT_LONG_OUT)
print("Wrote:", RT_WIDE_OUT)


Wrote: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/cache/rt_state_weekly.csv
Wrote: /Users/benjamincristol/Desktop/dissertation/argo_seir_eakf/cache/Rt_weekly_wide.csv


In [8]:
# ==========================================================
# 8. Optional QA summary
# ==========================================================

print("Rt summary by state:")
display(rt_long.groupby("state")["rt"].agg(["count", "mean", "min", "max"]).round(3).head(15))

print("Overall Rt range:", round(rt_long["rt"].min(), 3), "to", round(rt_long["rt"].max(), 3))


Rt summary by state:


,count,mean,min,max
state,,,,
AK,327,2.307,0.300,3.5
AL,345,2.299,0.300,3.5
AR,335,2.297,0.300,3.5
AZ,338,2.337,0.300,3.5
CA,327,2.299,0.314,3.5
CO,333,2.273,0.300,3.5
CT,332,2.296,0.300,3.5
DE,332,2.281,0.304,3.5
GA,341,2.301,0.300,3.5


Overall Rt range: 0.3 to 3.5


## 9. Required local data files for a clean GitHub repo

Minimum files needed to run this notebook from cached finalized inputs:

1. `state_preds_argo_step2_long.csv` or `state_preds_argo_step2.csv` — final weekly ARGO(X) step-2 predictions.
2. `ah_daily_allstates.csv` — daily state-level absolute humidity, wide format with `date` plus state abbreviation columns.
3. `ah_weekly_WMON_aligned.csv` — weekly Monday-aligned AH file. Optional if daily AH is complete, but useful as fallback/QA.

Outputs produced by this notebook:

1. `rt_state_weekly.csv` — long format: `state`, `date`, `rt`.
2. `Rt_weekly_wide.csv` — wide format: weekly dates by state columns.

Recommended repo structure:

```text
influenza-seirs-eakf-rt/
├── notebooks/
│   └── 01_reproduce_argox_seirs_eakf_rt.ipynb
├── data/
│   ├── state_preds_argo_step2_long.csv
│   ├── ah_daily_allstates.csv
│   └── ah_weekly_WMON_aligned.csv
├── outputs/
│   ├── rt_state_weekly.csv
│   └── Rt_weekly_wide.csv
└── README.md
```


## 10. Future validation notebook idea: ARGO(X) step-2 vs raw weekly ILI

A separate validation file can justify the ARGO(X) preprocessing by comparing weekly ARGO(X) step-2 predictions against raw FluView ILI over the same state-week grid.

Best approach:

1. Build a state-week panel with raw FluView ILI, ARGO(X) step-2 prediction, date, state, season, and MMWR week.
2. Compare ARGO(X) to a naive baseline such as lag-1 persistence or seasonal-average ILI, rather than only comparing to raw ILI itself.
3. Summarize by state-season using MAE, RMSE, peak week error, peak intensity error, and correlation.
4. Report paired state-season improvement: `100 * (baseline_error - argo_error) / baseline_error`.
5. Plot improvement distributions across state-seasons and bootstrap 95% confidence intervals.

The clean framing is: **ARGO(X) provides a smoother/nowcasted observation stream for assimilation relative to a naive raw-ILI baseline.**
